# 02. Checkpoint 3 model comparison

**Smart Recycling Coach** · COMP 365. Trains MobileNetV3-Large (E2), Large + oversampling (E4), MobileNetV3-Small (E5) and EfficientNet-B0 (E6) with the same split (seed 365) and recipe, compares them with the majority-class baseline (E0) on **validation only**, picks the best by val macro F1, then scores that one model on the **test split once**.

1. **Runtime > Change runtime type > T4 GPU.**
2. **Runtime > Run all.** About 15–25 minutes on a T4. Cell 2 asks to mount Google Drive.
3. Copy the two printed results blocks (validation, then test) into the script, slides and report. The last cell downloads the results zip and the final checkpoint so they can be committed.

All logic is in `src/recycle_coach/experiments.py`; this notebook only calls `scripts/run_cp3_experiments.py`.

In [ ]:
# 1. Get the code
import os, sys
from pathlib import Path
BRANCH = "main"   # change to "dakarai/cp3-experiments" if that branch isn't merged yet
REPO = Path("/content/insert-team-name-here")
if not REPO.exists():
    !git clone -q -b {BRANCH} https://github.com/rickeyjohnson/insert-team-name-here.git {REPO}
else:
    !cd {REPO} && git fetch -q && git checkout -q {BRANCH} && git pull -q
os.chdir(REPO)
!pip install -q -e . imagehash
sys.path.insert(0, str(REPO / "src"))
assert Path("scripts/run_cp3_experiments.py").exists(), "This branch has no CP3 runner; set BRANCH above."
!git log --oneline -1

In [ ]:
# 2. Get the images: copy data/processed/ from the team Drive, or rebuild it from TACO
import shutil, tempfile
import pandas as pd
from recycle_coach import config, ingest, clean

DRIVE_PROCESSED = "/content/drive/MyDrive/SmartRecyclingCoach-data/processed"  # edit if your Drive path differs
df_splits = pd.read_csv(config.SPLITS_CSV)
def missing_images():
    return [i for i in df_splits["image_id"] if not (config.PROCESSED_DIR / f"{i}.jpg").exists()]

if missing_images():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        if Path(DRIVE_PROCESSED).exists():
            print("Copying processed images from Drive...")
            shutil.copytree(DRIVE_PROCESSED, config.PROCESSED_DIR, dirs_exist_ok=True)
    except Exception as e:
        print("Drive not used:", e)
if missing_images():
    print(f"{len(missing_images())} images missing. Rebuilding crops from TACO (a few minutes)...")
    tmp = Path(tempfile.mkdtemp())
    ingest.ingest_taco_dataset(cap_per_class=450, output_csv=tmp / "images.csv")
    clean.run_cleaning(images_csv=tmp / "images.csv", output_clean_csv=tmp / "clean.csv")
missing = missing_images()
assert not missing, f"Still missing {len(missing)} images, e.g. {missing[:5]}"
print(f"All {len(df_splits)} images present.")
print(pd.crosstab(df_splits["final_label"], df_splits["split"])[["train", "val", "test"]])

## 3. Validation phase: train and compare every model (test split not touched)
Add `--resnet` to also train ResNet-18.

In [ ]:
!python scripts/run_cp3_experiments.py --phase val

In [ ]:
# 4. Comparison figure for the slides: val macro F1 and CPU time per image
import matplotlib.pyplot as plt
s = pd.read_csv(config.METRICS_DIR / "cp3_val_summary.csv")
names = {"baseline": "Baseline: always recycling", "mobilenet_v3_large": "MobileNetV3-L", "mobilenet_v3_small": "MobileNetV3-S",
         "efficientnet_b0": "EfficientNet-B0", "resnet18": "ResNet-18"}
s["label"] = [f"{names.get(a, a)}{' + oversample' if o else ''}\n({i})" for a, o, i in zip(s["arch"], s["oversample"], s["id"])]
colors = ["#9aa0a6" if a == "baseline" else "#2a78d6" for a in s["arch"]]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].barh(s["label"], s["val_macro_f1"], color=colors, height=0.6)
ax[0].axvline(config.TARGET_MACRO_F1, ls="--", color="#333", lw=1)
ax[0].text(config.TARGET_MACRO_F1 + 0.01, len(s) - 0.55, "target 0.70", fontsize=9, color="#333")
for y, v in enumerate(s["val_macro_f1"]): ax[0].text(max(v, config.TARGET_MACRO_F1) + 0.02 if abs(v - config.TARGET_MACRO_F1) < 0.06 else v + 0.01, y, f"{v:.2f}", va="center", fontsize=9)
ax[0].set_xlim(0, 1); ax[0].set_title(f"Validation macro F1 (n={int((df_splits['split']=='val').sum())})")
m = s[s["arch"] != "baseline"]
ax[1].barh(m["label"], m["cpu_ms_per_image"], color="#2a78d6", height=0.6)
for y, v in enumerate(m["cpu_ms_per_image"]): ax[1].text(v + 0.5, y, f"{v:.0f} ms", va="center", fontsize=9)
ax[1].set_title("CPU time per image (Colab CPU, batch 1)")
for a in ax: a.invert_yaxis(); a.spines[["top", "right"]].set_visible(False); a.grid(axis="x", alpha=0.25)
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "cp3_comparison.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for _, r in m.iterrows():
    h = pd.read_csv(config.METRICS_DIR / f"cp3_{r['id']}_history.csv")
    ax.plot(h["epoch"], h["val_macro_f1"], marker="o", ms=4, lw=2, label=r["label"].replace("\n", " "))
ax.axhline(s.loc[s["id"] == "E0", "val_macro_f1"].iloc[0], ls="--", color="#9aa0a6", label="Baseline (E0)")
ax.set_xlabel("Stage-2 epoch"); ax.set_ylabel("Val macro F1"); ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(alpha=0.25)
ax.set_title("Validation macro F1 by epoch")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "cp3_val_curves.png", dpi=150); plt.show()

## 5. Test phase: the val pick, on the test split, once
The script refuses to run a second time. Don't re-run this cell after changing anything.

In [ ]:
!python scripts/run_cp3_experiments.py --phase test

## 6. Optional: campus photo test
Upload the team's photos (named like `test_js_special_handling_battery.jpg`) to a `campus_test/` folder in the Files pane, then run this. About 15 photos, 5 per class, are needed to judge the 9/10 and 8/10-per-class targets.

In [ ]:
if Path("campus_test").exists():
    !python scripts/run_demo_test.py campus_test/
else:
    print("No campus_test/ folder; skipping.")

## 7. Download results and the final checkpoint (commit both)

In [ ]:
zip_path = shutil.make_archive("/content/cp3_results", "zip", root_dir=config.ROOT, base_dir="results")
try:
    from google.colab import files
    for p in (zip_path, str(config.CHECKPOINT_PATH)):
        files.download(p)
except ImportError:
    print("Results:", zip_path, "| checkpoint:", config.CHECKPOINT_PATH)